In [ ]:
# =============================================================================
# Resolution Check Script for PACE-VCF Pipeline
# =============================================================================
# Checks geographic resolution at each processing step for tile h12v09

import numpy as np
from pathlib import Path
import rasterio
import netCDF4 as nc
from osgeo import gdal

# =============================================================================
# CONFIGURATION
# =============================================================================

TILE = "h12v09"
YEAR = 2025

# Directories
PACE_VCF_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF")
OUTPUT_BASE = PACE_VCF_BASE / "output" / TILE / str(YEAR)

# Input data directories
PACE_SFREFL_DIR = PACE_VCF_BASE / "PACE_SFREFL_DAILY"
PACE_SFREFL_8DAY_DIR = PACE_VCF_BASE / "PACE_SFREFL"
VIIRS_DIR = PACE_VCF_BASE / "VIIRS_VNP21A1D"

# Processing step outputs
COMPOSITES_DIR = OUTPUT_BASE / "2-Composites"
METRICS_DIR = OUTPUT_BASE / "3-Metrics"

# MODIS sinusoidal parameters
MODIS_TILE_SIZE_M = 1111950.5196666666

print("="*70)
print(f"RESOLUTION CHECK: Tile {TILE}")
print("="*70)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def check_geotiff(filepath, description):
    """Check resolution of a GeoTIFF file."""
    print(f"\n{description}")
    print("-" * 50)
    
    if not filepath.exists():
        print(f"  File not found: {filepath}")
        return None
    
    try:
        with rasterio.open(filepath) as src:
            width, height = src.width, src.height
            res_x, res_y = src.res
            bounds = src.bounds
            crs = src.crs
            
            extent_x = bounds.right - bounds.left
            extent_y = bounds.top - bounds.bottom
            
            print(f"  File: {filepath.name}")
            print(f"  Dimensions: {width} x {height} pixels")
            print(f"  Pixel size: {res_x:.2f} x {res_y:.2f} meters")
            print(f"  Pixel size: {res_x/1000:.4f} x {res_y/1000:.4f} km")
            print(f"  Tile extent: {extent_x/1000:.2f} x {extent_y/1000:.2f} km")
            print(f"  Bounds: {bounds}")
            print(f"  CRS: {crs}")
            print(f"  Bands: {src.count}")
            
            return {
                'width': width, 'height': height,
                'res_x': res_x, 'res_y': res_y,
                'extent_x': extent_x, 'extent_y': extent_y
            }
    except Exception as e:
        print(f"  Error reading file: {e}")
        return None


def check_binary_file(filepath, expected_size, description):
    """Check a binary file and infer resolution."""
    print(f"\n{description}")
    print("-" * 50)
    
    if not filepath.exists():
        print(f"  File not found: {filepath}")
        return None
    
    try:
        data = np.fromfile(filepath, dtype=np.int16)
        total_pixels = len(data)
        
        sqrt_size = int(np.sqrt(total_pixels))
        if sqrt_size * sqrt_size == total_pixels:
            width = height = sqrt_size
        else:
            width = height = expected_size
            
        pixel_size_m = MODIS_TILE_SIZE_M / width
        
        print(f"  File: {filepath.name}")
        print(f"  Total values: {total_pixels:,}")
        print(f"  Inferred dimensions: {width} x {height} pixels")
        print(f"  Inferred pixel size: {pixel_size_m:.2f} meters")
        print(f"  Inferred pixel size: {pixel_size_m/1000:.4f} km")
        print(f"  File size: {filepath.stat().st_size / 1024:.1f} KB")
        
        return {'width': width, 'height': height, 'pixel_size_m': pixel_size_m}
    except Exception as e:
        print(f"  Error reading file: {e}")
        return None


def check_netcdf(filepath, description):
    """Check resolution of a NetCDF file (PACE L3m)."""
    print(f"\n{description}")
    print("-" * 50)
    
    if not filepath.exists():
        print(f"  File not found: {filepath}")
        return None
    
    try:
        with nc.Dataset(filepath, 'r') as ds:
            lat = ds.variables.get('lat', ds.variables.get('latitude', None))
            lon = ds.variables.get('lon', ds.variables.get('longitude', None))
            
            if lat is not None and lon is not None:
                lat_vals = lat[:]
                lon_vals = lon[:]
                
                lat_res = np.abs(np.diff(lat_vals).mean()) if len(lat_vals) > 1 else None
                lon_res = np.abs(np.diff(lon_vals).mean()) if len(lon_vals) > 1 else None
                
                print(f"  File: {filepath.name}")
                print(f"  Dimensions: {len(lon_vals)} x {len(lat_vals)} (lon x lat)")
                
                if lat_res and lon_res:
                    print(f"  Lat resolution: {lat_res:.6f} deg ({lat_res * 111:.2f} km at equator)")
                    print(f"  Lon resolution: {lon_res:.6f} deg ({lon_res * 111:.2f} km at equator)")
                
                print(f"  Lat range: {lat_vals.min():.2f} to {lat_vals.max():.2f}")
                print(f"  Lon range: {lon_vals.min():.2f} to {lon_vals.max():.2f}")
                
                if hasattr(ds, 'spatialResolution'):
                    print(f"  Stated resolution: {ds.spatialResolution}")
                
                data_vars = [v for v in ds.variables if v not in ['lat', 'lon', 'latitude', 'longitude', 'time']]
                print(f"  Data variables: {len(data_vars)}")
                
                return {'lat_res': lat_res, 'lon_res': lon_res, 
                        'n_lat': len(lat_vals), 'n_lon': len(lon_vals)}
            else:
                print(f"  Could not find lat/lon variables")
                print(f"  Available variables: {list(ds.variables.keys())}")
                return None
    except Exception as e:
        print(f"  Error reading file: {e}")
        return None


def check_viirs_hdf(filepath, description):
    """Check resolution of VIIRS HDF5 file."""
    print(f"\n{description}")
    print("-" * 50)
    
    if not filepath.exists():
        print(f"  File not found: {filepath}")
        return None
    
    try:
        ds = gdal.Open(str(filepath))
        if ds is None:
            print(f"  Could not open with GDAL")
            return None
        
        subdatasets = ds.GetSubDatasets()
        print(f"  File: {filepath.name}")
        print(f"  Subdatasets: {len(subdatasets)}")
        
        if subdatasets:
            for name, desc in subdatasets[:3]:
                sub_ds = gdal.Open(name)
                if sub_ds:
                    gt = sub_ds.GetGeoTransform()
                    width, height = sub_ds.RasterXSize, sub_ds.RasterYSize
                    
                    short_desc = desc[:50] + "..." if len(desc) > 50 else desc
                    print(f"\n  Subdataset: {short_desc}")
                    print(f"    Dimensions: {width} x {height}")
                    if gt:
                        print(f"    Pixel size: {gt[1]:.2f} x {abs(gt[5]):.2f}")
                    sub_ds = None
        
        ds = None
        return True
    except Exception as e:
        print(f"  Error reading file: {e}")
        return None


# =============================================================================
# CHECK EACH PROCESSING STEP
# =============================================================================

results = {}

# -----------------------------------------------------------------------------
# 1. PACE Surface Reflectance (L3m input)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("1. PACE SURFACE REFLECTANCE (L3m Input)")
print("="*70)

pace_files = []
for pattern in ["*.nc", "PACE_OCI*.nc"]:
    pace_files.extend(list(PACE_SFREFL_8DAY_DIR.glob(pattern)))
    if not pace_files:
        pace_files.extend(list(PACE_SFREFL_DIR.glob(f"**/{pattern}")))

if pace_files:
    results['pace_l3m'] = check_netcdf(pace_files[0], "PACE L3m Surface Reflectance")
else:
    print(f"\n  No PACE L3m files found in {PACE_SFREFL_DIR}")

# -----------------------------------------------------------------------------
# 2. VIIRS LST (Input)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("2. VIIRS LST (Input)")
print("="*70)

viirs_patterns = [f"*{TILE}*.h5", f"*{TILE}*.hdf", f"VNP21*{TILE}*.h5"]
viirs_files = []
for pattern in viirs_patterns:
    viirs_files.extend(list(VIIRS_DIR.glob(f"**/{pattern}")))

if viirs_files:
    results['viirs'] = check_viirs_hdf(viirs_files[0], "VIIRS VNP21 LST")
else:
    print(f"\n  No VIIRS files found for tile {TILE} in {VIIRS_DIR}")

# -----------------------------------------------------------------------------
# 3. Composites (Intermediate output)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("3. COMPOSITES (Intermediate Output)")
print("="*70)

wavelength_dir = COMPOSITES_DIR / "wavelengths"
if wavelength_dir.exists():
    wl_files = sorted(wavelength_dir.glob("*.bin"))
    if wl_files:
        results['composite_wl'] = check_binary_file(wl_files[0], 600, "Wavelength Composite (.bin)")

agg_dir = COMPOSITES_DIR / "aggregated"
if agg_dir.exists():
    agg_files = sorted(agg_dir.glob("*.bin"))
    if agg_files:
        results['composite_agg'] = check_binary_file(agg_files[0], 600, "Aggregated Band Composite (.bin)")

thermal_dir = COMPOSITES_DIR / "thermal"
if thermal_dir.exists():
    therm_files = sorted(thermal_dir.glob("*.bin"))
    if therm_files:
        results['composite_thermal'] = check_binary_file(therm_files[0], 600, "Thermal Composite (.bin)")

composite_tifs = list(COMPOSITES_DIR.glob("*.tif"))
if composite_tifs:
    results['composite_tif'] = check_geotiff(composite_tifs[0], "Composite GeoTIFF")

# -----------------------------------------------------------------------------
# 4. Metrics (Final output)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("4. METRICS (Final Output)")
print("="*70)

modis_metrics = METRICS_DIR / "MODIS_Metrics.tif"
results['metrics_modis'] = check_geotiff(modis_metrics, "MODIS_Metrics.tif")

pace_metrics = METRICS_DIR / "PACE_Metrics.tif"
results['metrics_pace'] = check_geotiff(pace_metrics, "PACE_Metrics.tif")

altsort_metrics = METRICS_DIR / "PACE_AltSort_Metrics.tif"
results['metrics_altsort'] = check_geotiff(altsort_metrics, "PACE_AltSort_Metrics.tif")

# -----------------------------------------------------------------------------
# 5. Training Data Reference (MODIS VCF)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("5. REFERENCE: MODIS TRAINING DATA")
print("="*70)

MODIS_TRAINING_DIR = Path("/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C/training")
training_files = list(MODIS_TRAINING_DIR.glob(f"{TILE}*.parq"))

if training_files:
    import pandas as pd
    print(f"\nMODIS Training Parquet")
    print("-" * 50)
    df = pd.read_parquet(training_files[0])
    print(f"  File: {training_files[0].name}")
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  x range: {df['x'].min()} to {df['x'].max()}")
    print(f"  y range: {df['y'].min()} to {df['y'].max()}")
    
    unique_x = df['x'].nunique()
    unique_y = df['y'].nunique()
    print(f"  Unique x: {unique_x}, Unique y: {unique_y}")
    print(f"  Implied grid: {unique_x} x {unique_y}")
    
    df['pace_x'] = df['x'] // 8
    df['pace_y'] = df['y'] // 8
    unique_pace_x = df['pace_x'].nunique()
    unique_pace_y = df['pace_y'].nunique()
    print(f"  After //8 aggregation: {unique_pace_x} x {unique_pace_y} PACE pixels")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*70)
print("RESOLUTION SUMMARY")
print("="*70)

print(f"\n{'Data Product':<35} {'Dimensions':<20} {'Pixel Size':<20}")
print("-"*75)

if results.get('pace_l3m'):
    print(f"{'PACE L3m (input)':<35} {'Global 2km grid':<20} {'~2 km (geographic)':<20}")

print(f"{'VIIRS VNP21 (input)':<35} {'1200x1200 (1km)':<20} {'~1 km':<20}")

if results.get('composite_wl'):
    r = results['composite_wl']
    dims = f"{r['width']}x{r['height']}"
    pix = f"{r['pixel_size_m']/1000:.3f} km"
    print(f"{'Wavelength composites':<35} {dims:<20} {pix:<20}")

if results.get('composite_agg'):
    r = results['composite_agg']
    dims = f"{r['width']}x{r['height']}"
    pix = f"{r['pixel_size_m']/1000:.3f} km"
    print(f"{'Aggregated composites':<35} {dims:<20} {pix:<20}")

if results.get('metrics_modis'):
    r = results['metrics_modis']
    dims = f"{r['width']}x{r['height']}"
    pix = f"{r['res_x']/1000:.3f} km"
    print(f"{'MODIS_Metrics.tif':<35} {dims:<20} {pix:<20}")

if results.get('metrics_pace'):
    r = results['metrics_pace']
    dims = f"{r['width']}x{r['height']}"
    pix = f"{r['res_x']/1000:.3f} km"
    print(f"{'PACE_Metrics.tif':<35} {dims:<20} {pix:<20}")

print(f"{'MODIS Training (reference)':<35} {'4800x4800':<20} {'~0.232 km (250m)':<20}")
print(f"{'Training aggregated to PACE':<35} {'600x600':<20} {'~1.853 km':<20}")

print("\n" + "="*70)
print("KEY CALCULATION")
print("="*70)
print(f"""
MODIS Tile Size: {MODIS_TILE_SIZE_M:,.2f} meters ({MODIS_TILE_SIZE_M/1000:.2f} km)

At 600x600 pixels:
  Pixel size = {MODIS_TILE_SIZE_M/1000:.2f} km / 600 = {MODIS_TILE_SIZE_M/600/1000:.4f} km = {MODIS_TILE_SIZE_M/600:.2f} m

At 4800x4800 pixels (MODIS 250m):
  Pixel size = {MODIS_TILE_SIZE_M/1000:.2f} km / 4800 = {MODIS_TILE_SIZE_M/4800/1000:.4f} km = {MODIS_TILE_SIZE_M/4800:.2f} m

Aggregation factor: 4800 / 600 = 8 (so 8x8 = 64 MODIS pixels per PACE pixel)
""")

print("="*70)